# Creating Cherry Rainbow Tables

For N = 2**16, currently only making one table

## Imports

In [233]:
import pickle
import random
from hashlib import sha256
import mmh3
from tqdm import tqdm
from math import pi, sqrt, e, log

## Table Parameters

### Initialise Startpoints

In [234]:
# initialise startpoints - either generate them or load from pickle - to keep same across runs
def get_startpoints(N, m_0, nlabel, alpha):
    # try opening pickle file, else generate and save
    try:
        with open(f'startpoints_N_{nlabel}_alpha_{alpha}.pkl', 'rb') as f:
            startpoints = pickle.load(f)

    # no file found - generate and save
    except FileNotFoundError:
        # random but unique - store as a set?
        startpoints = set()
        while len(startpoints) < m_0:
            startpoints.add(random.randint(0, N-1))

        # store startpoints in pickle file
        with open(f'startpoints_N_{nlabel}_alpha_{alpha}.pkl', 'wb') as f:
            pickle.dump(startpoints, f)

    # return the startpoints
    return startpoints

### Parameters

In [235]:
# label N to find easier - label is the exponent
nlabel = 16
##############################################################################
N = 2 ** 16 # keyspace
p = 1 - e ** -2 # our table coverage - 86%
##############################################################################
t = round(log(1-p)/log(1-N**(-1/3))) # chain length t
alpha = 0.95 # maximality factor
mt_target = N**(2/3) # our target mt
m_0 = round(mt_target/(1-alpha))    # m_0 - number of startpoints
##############################################################################
# initialise startpoints 
startpoints = get_startpoints(N, m_0, nlabel, alpha)


### Cherry-picks per column - Kis

In [236]:
# get # cherry-picks per column from pickle file 

# load pickle file
with open(f'higher_costs_ftol_1.pickle', 'rb') as f:
    data = pickle.load(f)

# structure of file
# array of different alpha used
    # for each alpha, array of different costs used
    # for each cost, array arranged as [Kjs, m_0, final_cost, m_values]

# to get Kjs for alpha = 0.95 and cost factor = 5
Kis = data[-1][0][0]

## Hash and Reduction Functions

In [237]:
# # Hash function
# def H(x):
# 	return int.from_bytes(sha256(x.to_bytes(8)).digest())

# # Reduction function
# # currently mod but should change to murmurhash in future
# # def r(y, i, ell=0):   # also takes in ell - number of tables - for future use (but currently ell=0)
# # 	return (y + i + ell*t) % N

# def r(y, i, ell=0):
# 	# use index as seed and do mod N at the end
# 	return mmh3.hash(y.to_bytes(32, 'little'), i + ell*t, signed=False) % N

def H(x):
    return sha256(x.to_bytes(8, 'little')).digest()  # return bytes directly

def r(y, i, ell=0):
    return mmh3.hash(y, i + ell*t, signed=False) % N


## Building the Table

In [238]:
# take in m_0 and t as parameters - how many chains to start with and how long to make the chains
# store the table as a dictionary of endpoint:startpoint pairs (rather than sp:ep for easier lookup later)
# store in a pickle file
# need to also store the reduction function used in each column - how much more memory is taken up? only need to store index per column, so t more bits of memory

def build_cherry_table_old(t, alpha, startpoints, Kis):
    # store table in dictionary
    table = {}  # store all the points then remove duplicate entries - can't do duplicate keys in dictionary anyway so we can just store all ep:sp

    # Instead of making the table chain by chain, we have to make it column by column to test what reduction function to choose
    # store reduction function indexes
    rf_indexes = []

    # initialise table wit SP:SP pairs
    for point in startpoints:
        table[point] = point

    # hash all current points
    hashed_points = [H(sp) for sp in startpoints]    

    """
    - for each column
        - hash the current points
        - get the # cherry-picks for that column
        - use each reduction function on the points, and store the best running m_i+1 and reduction function index
            - is it quicker to store the reduced points and overwrite them each time, or tally the points as we test and then reduce all points at the end with best r - but then we're doing m_i more reductions?
        - for each rf sample:
            - reduce the column and store in a set
            - get the length of the set - if its better than before then note down this rf 
            - clear the set and do it again
        - with our best rf, 
    """

    # for each column in the table
    for i in range(t):

        # get # cherry-picks for this column
        k_i = round(Kis[i])

        # variables to store best reduction function info
        best_m_i_plus_1 = -1
        best_rf_index = -1

        # set to store reduced points for each rf test
        reduced_points_set = set()

        # test each reduction function
        for rf_index in range(k_i):
            # reduce all hashed points with this reduction function
            for hp in hashed_points:
                rp = r(hp, rf_index)  # reduce point
                reduced_points_set.add(rp)   # add to set

            # get m_i+1
            m_i_plus_1 = len(reduced_points_set)

            # check if best
            if m_i_plus_1 > best_m_i_plus_1:
                best_m_i_plus_1 = m_i_plus_1
                best_rf_index = rf_index

            # clear set for next rf test
            reduced_points_set.clear()

        # with best rf, reduce all hashed points and update for next column

        # make a current table to now have m_i:SP pairs
        current_column = dict()
        # for each (key, value) pair in the current table, make its new entry
            # i don't want to hash again - can i reuse hashes_points? how do I know its in order?
        for (key, sp), hp in zip(table.items(), hashed_points):
            ep = r(hp, best_rf_index)  # get endpoint with best rf
            current_column[ep] = sp  # store m_i:SP pair

        # update table to current column
        table = current_column

        # update hashed_points
        hashed_points = [H(m_i) for m_i in table.keys()]

        # store the best rf index
        rf_indexes.append(best_rf_index)
        
    # store the table and rf_indexes in a pickle file
    # with open(f'cherry_table_alpha_{alpha}_t_{t}.pkl', 'wb') as f:
    #     pickle.dump((table), f)

    # with open(f'cherry_rf_alpha_{alpha}_t_{t}.pkl', 'wb') as f:
    #     pickle.dump((table), f)


    return table

In [239]:
# build cherry table
def build_cherry_table(t, alpha, startpoints, Kis):
    # store table in dictionary
    # initialise the table with sp:sp pairs
    table = {sp: sp for sp in startpoints}  # store all the points then remove duplicate entries - can't do duplicate keys in dictionary anyway so we can just store all ep:sp

    # Instead of making the table chain by chain, we have to make it column by column to test what reduction function to choose
    # store reduction function indexes
    rf_indexes = []

    # for each column
    for i in tqdm(range(t), desc=f"Calculating columns: "):

        # variable to store best cherry-pick
        best_trial = -1

        # hash all current points then store with startpoints - this stores all our current points
        hashed_points = {H(mi): sp for mi, sp in table.items()}

        # we are going to continuously replace table with the best rf trial, so we empty it for now
            # we haven't lost the current points as we have them hashed in the hashed_points dictionary
        table = {}

        # get # cherry-picks for this column
        k_i = round(Kis[i])

        # trial all the reduction functions for this column
        # for rf_trial in tqdm(range(k_i), desc=f"Choosing best RF: "):
        for rf_trial in range(k_i):

            # create a trial column to store results of current trial
            trial_column = {}

            # go through each key in hashed_points and store its reduction with sp
            for x in hashed_points:
                # reduce the hash and store in column
                trial_column[r(x, rf_trial)] = hashed_points[x]

            # if trial_column is bigger than current table stored, we replace it
            if len(trial_column) > len(table):
                # replace it 
                table = trial_column
                # replace best cherry-pick
                best_trial = rf_trial

        # now store the best rf cherry pick
        rf_indexes.append(best_trial)

    # finished, so return table and rf indexes
    return table, rf_indexes



## Run

### Precomputation Phase - Build the table

In [240]:
# either build or load table
def get_cherry_table():
    # try loading table from pickle file
    try:
        with open(f'cherry_table_alpha_{alpha}_t_{t}.pkl', 'rb') as f:
            table = pickle.load(f)

    # if no pickle file found, build the table
    except FileNotFoundError:
        table = build_cherry_table(t, alpha, startpoints, Kis)

    return table


In [241]:
table = get_cherry_table()

Calculating columns:   0%|          | 0/80 [00:00<?, ?it/s]

Calculating columns: 100%|██████████| 80/80 [05:07<00:00,  3.85s/it]


In [242]:
points, indexes = table

In [248]:
len(points)

2197

In [244]:
startpoints

{0,
 1,
 2,
 5,
 6,
 9,
 10,
 14,
 15,
 16,
 22,
 24,
 25,
 27,
 28,
 29,
 30,
 31,
 33,
 36,
 38,
 39,
 41,
 42,
 45,
 48,
 49,
 50,
 59,
 61,
 62,
 64,
 65,
 67,
 68,
 71,
 78,
 80,
 84,
 85,
 86,
 87,
 88,
 90,
 91,
 92,
 94,
 95,
 96,
 98,
 99,
 102,
 105,
 107,
 108,
 111,
 112,
 113,
 117,
 120,
 124,
 128,
 130,
 133,
 137,
 141,
 143,
 144,
 146,
 147,
 148,
 149,
 150,
 152,
 153,
 155,
 156,
 161,
 164,
 165,
 167,
 169,
 170,
 171,
 172,
 173,
 177,
 178,
 179,
 182,
 184,
 185,
 186,
 188,
 189,
 190,
 191,
 192,
 197,
 198,
 200,
 201,
 202,
 203,
 205,
 206,
 208,
 213,
 214,
 215,
 217,
 219,
 222,
 225,
 226,
 232,
 233,
 236,
 237,
 238,
 241,
 243,
 244,
 246,
 248,
 249,
 252,
 254,
 255,
 256,
 257,
 259,
 261,
 262,
 266,
 271,
 274,
 275,
 278,
 279,
 280,
 283,
 284,
 287,
 288,
 289,
 290,
 293,
 299,
 305,
 306,
 307,
 308,
 312,
 314,
 316,
 317,
 318,
 319,
 323,
 326,
 327,
 329,
 330,
 331,
 332,
 333,
 334,
 335,
 336,
 337,
 340,
 342,
 344,
 352,
 353,
 

In [245]:
for item in Kis:
    print(round(item))

1
677
806
921
1027
1124
1214
1299
1379
1454
1525
1594
1659
1721
1781
1838
1894
1946
1998
2048
2096
2142
2188
2232
2275
2317
2358
2397
2436
2474
2511
2547
2582
2617
2651
2684
2717
2748
2780
2811
2841
2871
2900
2929
2957
2985
3012
3039
3066
3093
3118
3144
3169
3195
3219
3243
3267
3291
3314
3337
3360
3383
3406
3428
3449
3472
3493
3514
3536
3557
3577
3598
3618
3638
3659
3678
3699
3718
3737
3757


In [246]:
with open(f'old_cherry_table.pkl', 'rb') as f:
    oldtable = pickle.load(f)

In [247]:
oldtable

{43177: 57844,
 12169: 63259,
 30483: 60716,
 15965: 63787,
 14566: 60878,
 17677: 55073,
 4013: 58167,
 56694: 29724,
 26234: 62117,
 47254: 52183,
 35280: 37975,
 8001: 47618,
 15978: 52878,
 61286: 54540,
 36234: 18058,
 31320: 31258,
 25872: 41956,
 43771: 54285,
 38493: 14222,
 38510: 55697,
 28907: 44112,
 21002: 33555,
 1840: 65517,
 14354: 32099,
 18914: 57205,
 28038: 56272,
 46426: 61514,
 32115: 60327,
 35060: 63369,
 49378: 53186,
 58499: 48124,
 47951: 54150,
 35567: 61966,
 34373: 52532,
 5781: 29750,
 45962: 37509,
 63342: 32721,
 15018: 29354,
 41798: 8162,
 33559: 41649,
 15328: 60658,
 34041: 51100,
 39882: 36453,
 19074: 14696,
 43336: 41526,
 7241: 37257,
 18234: 63829,
 3028: 63182,
 18639: 26244,
 27488: 43597,
 15042: 36751,
 9783: 64774,
 45986: 56334,
 22273: 8754,
 12191: 39212,
 2286: 57056,
 43153: 63924,
 27866: 55974,
 16744: 42728,
 26866: 8572,
 10664: 40707,
 60389: 34263,
 58782: 46150,
 7220: 54602,
 50242: 27550,
 40096: 48062,
 37030: 49287,
 17713: